<a href="https://colab.research.google.com/github/Shikher-jain/FastApi/blob/main/Insurance_Project/fastapi_ml_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.metrics import classification_report, accuracy_score

In [6]:
df = pd.read_csv('/insurance.csv')

In [7]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
6,19,80.1,1.68,3.590000,True,Hyderabad,student,Medium
7,31,105.7,1.78,10.865821,True,Delhi,government_job,Medium
63,47,71.3,1.82,41.660000,True,Gaya,business_owner,High
53,41,101.3,1.85,30.000000,True,Delhi,government_job,Medium
56,24,101.9,1.55,2.860000,True,Kolkata,student,Medium


In [9]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [10]:
df_feat = df.copy()

In [12]:
df_feat['bmi'] = df_feat['weight']/(df_feat['height'])**2

In [13]:
df_feat.sample()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi
92,37,62.7,1.85,30.0,True,Lucknow,government_job,Low,18.319942


In [14]:
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    return "senior"

In [15]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [17]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group
0,67,119.8,1.56,2.920000,False,Jaipur,retired,High,49.227482,senior
23,35,70.3,1.78,23.710000,False,Mysore,unemployed,Medium,22.187855,adult
3,22,109.4,1.55,3.340000,True,Mumbai,student,Medium,45.535900,young
54,75,54.5,1.61,3.320000,True,Lucknow,retired,High,21.025423,senior
94,50,105.4,1.78,10.542289,False,Bangalore,government_job,Low,33.266002,middle_aged


In [18]:
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [19]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [20]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk
25,59,60.2,1.55,30.00,False,Mysore,government_job,Low,25.057232,middle_aged,low
85,33,51.4,1.86,34.66,False,Chennai,private_job,Low,14.857209,adult,low
86,35,66.0,1.89,37.38,False,Hyderabad,freelancer,Low,18.476526,adult,low
47,55,116.4,1.87,8.34,False,Chandigarh,private_job,Medium,33.286625,middle_aged,medium
43,72,85.7,1.71,1.56,False,Chennai,retired,Medium,29.308163,senior,medium


In [21]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [22]:
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [24]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [40]:
df_feat.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
86,35,66.0,1.89,37.38000,False,Hyderabad,freelancer,Low,18.476526,adult,low,1
26,33,79.0,1.61,23.61000,False,Jaipur,freelancer,Medium,30.477219,adult,medium,2
34,75,100.8,1.75,0.68000,True,Kota,retired,High,32.914286,senior,high,3
78,52,95.6,1.85,14.74000,True,Jalandhar,freelancer,High,27.932798,middle_aged,medium,2
99,40,70.0,1.59,28.16664,True,Bangalore,government_job,Low,27.688778,adult,medium,1


In [41]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
52,2.96,student,47.344720,young,medium,2,Medium
90,21.07,business_owner,21.093750,middle_aged,low,1,Low
19,2.79,student,43.437500,young,high,2,High
6,3.59,student,28.380102,young,medium,1,Medium
40,40.19,unemployed,24.349609,adult,medium,1,Medium


In [42]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [43]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,medium,2,2.92000,retired
1,30.189017,adult,medium,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,medium,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,medium,1,34.01000,private_job
97,18.765432,middle_aged,low,1,44.86000,freelancer
98,30.521676,adult,medium,1,28.30000,business_owner


In [44]:
y

,insurance_premium_category
0,High
1,Low
2,Low
3,Medium
4,High
...,...
95,Low
96,Low
97,Low
98,Low


In [45]:
# Define categorical and numeric features
categorical_features = ["age_group", "lifestyle_risk", "occupation", "city_tier"]
numeric_features = ["bmi", "income_lpa"]

In [46]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        # ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)
#   handle_unknown="ignore" → Prevents errors if new categories appear in test
#   data.sparse_output=False → Returns a dense array (easier for many models and debugging).

In [47]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])


In [48]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [49]:
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.9

In [50]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
31,15.258742,adult,medium,2,11.77,private_job
52,47.344720,young,medium,2,2.96,student
51,38.827923,middle_aged,high,2,28.95,private_job
10,22.949982,adult,medium,1,32.78,business_owner
65,37.662982,middle_aged,high,2,38.07,unemployed


In [53]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)

In [52]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed